# Aula 7 — Lakehouse na prática com DuckDB

> **Objetivo:** organizar o pipeline da Aula 6 em camadas bronze, silver e gold, usando Parquet e SQL — com uma engine que roda no seu notebook.
> **Duração:** 3 horas · **Pré-requisito:** Aulas 1 a 6

---

## O problema

Na Aula 6 você construiu um pipeline completo:

`CSV bruto → limpeza → validação → JSON`

Funciona. Mas em um cenário real aparecem três perguntas que esse código não responde:

1. **Onde ficam os dados?** Bruto, limpo e agregado misturados na mesma pasta não escala.
2. **E quando chegar o arquivo de amanhã?** O de hoje é sobrescrito? Duplicado?
3. **E se o arquivo tiver 50 milhões de linhas?** Ler tudo com `for` em Python fica inviável.

O padrão que o mercado adotou para responder as três se chama **Lakehouse**. É isso que vamos construir hoje.

---

## O que é um Lakehouse

Antes dele, existiam dois mundos separados:

| | Data Warehouse | Data Lake |
|---|---|---|
| Guarda | tabelas organizadas | arquivos soltos |
| Custo | caro | barato |
| Aceita dado não estruturado | ❌ | ✅ |
| Tem schema e SQL | ✅ | ❌ (vira "pântano") |
| Governança | forte | fraca |

> Um **Lakehouse** é um Data Lake (arquivos baratos) que ganhou as qualidades de um Data Warehouse (schema, SQL, transações).

Na prática, três ideias:

| Ideia | O que significa |
|---|---|
| **Arquivos, não servidor** | Os dados vivem como arquivos (Parquet) em uma pasta ou bucket |
| **Camadas** | bronze (bruto) → silver (limpo) → gold (pronto para o negócio) |
| **SQL por cima** | Uma engine lê esses arquivos e responde SQL, sem carregar nada para um banco |

### As três camadas

| Camada | Regra | Quem usa |
|---|---|---|
| 🥉 **bronze** | dado **exatamente como chegou**, sem limpeza. Só muda o formato. | engenharia de dados |
| 🥈 **silver** | limpo, tipado, validado, sem duplicata. Uma linha por entidade. | analistas, cientistas |
| 🥇 **gold** | agregado e pronto para responder uma pergunta de negócio. | dashboards, diretoria |

**Por que não limpar direto na entrada?** Porque quando a regra de negócio mudar — e ela vai mudar — você precisa reprocessar a partir do dado original. Se você limpou na entrada e jogou o bruto fora, o dado original se perdeu para sempre. A bronze é a sua rede de segurança.

## Onde o DuckDB entra

O DuckDB é uma engine SQL que roda **dentro do seu programa Python** — sem servidor, sem Docker, sem cluster. É comum descrevê-lo como "o SQLite da análise de dados".

Ele lê CSV, JSON e Parquet **direto do disco** com SQL. É o jeito mais simples que existe hoje de montar um Lakehouse — e o raciocínio é **o mesmo** de Spark + Delta Lake em produção.

## 1. Instalando e conectando

In [ ]:
# No Google Colab, instale o DuckDB (leva uns 10 segundos)
!pip install duckdb -q

import duckdb
print('DuckDB versão:', duckdb.__version__)

**📝 O que essa célula faz:**

- `!pip install duckdb -q` → o `!` diz ao notebook "isso não é Python, é um comando de terminal". `pip` é o instalador de bibliotecas, e `-q` é *quiet* (instala sem imprimir mil linhas).
- `import duckdb` → carrega a biblioteca, do mesmo jeito que você fez `import csv` e `import json`.

> 💡 Você instala **uma vez** por sessão do Colab.

In [ ]:
# Cria a conexão. Se o arquivo não existir, o DuckDB cria.
con = duckdb.connect('lakehouse.db')

print('Conectado ✓')

**📝 O que essa célula faz:**

Cria uma **conexão** com um banco DuckDB guardado no arquivo `lakehouse.db`.

Repare: um banco de dados inteiro aqui é **um único arquivo**. Sem servidor, sem usuário, sem senha, sem porta. É por isso que o DuckDB é tão prático.

**❓ E se eu quisesse um banco só na memória?** `duckdb.connect()` sem argumento cria um banco temporário, que some quando você fecha o notebook.

## 2. Landing zone — o arquivo que "chegou"

*Landing zone* é a área onde os arquivos **aterrissam** vindos da origem, antes de qualquer tratamento. É a porta de entrada do Lakehouse.

Vamos usar o **mesmo arquivo** do módulo de Python, com os mesmos problemas de qualidade: campos vazios, valor negativo e data inválida.

In [ ]:
import os

os.makedirs('landing', exist_ok=True)

dados_dia_20 = """id,nome,email,cidade,valor_compra,data_compra
1001,Ana Lima,ana@email.com,São Paulo,1850.00,20/01/2024
1002,Bruno Costa,,Rio de Janeiro,320.50,20/01/2024
1003,Carla Dias,carla@email.com,Belo Horizonte,4200.00,20/01/2024
1004,Diego Nunes,diego@email.com,Porto Alegre,-50.00,20/01/2024
1005,Eva Melo,eva@email.com,Recife,,20/01/2024
1006,Felipe Rosa,felipe@email.com,Fortaleza,890.00,20/01/2024
1007,,joao@email.com,Manaus,1200.00,20/01/2024
1008,Gisele Tuno,gisele@email.com,Salvador,670.00,data_invalida
"""

with open('landing/pedidos_20240120.csv', 'w', encoding='utf-8') as f:
    f.write(dados_dia_20)

print('✓ landing/pedidos_20240120.csv criado (8 registros)')

**📝 O que essa célula faz:**

Exatamente o que você já viu no módulo de arquivos: `os.makedirs` cria a pasta, `with open(...) as f` abre para escrita e `f.write(...)` grava.

O `"""` (três aspas) permite escrever um texto de várias linhas em uma variável só.

## 3. SQL direto no arquivo

Primeiro truque do DuckDB: escreva `FROM 'arquivo.csv'` como se o arquivo fosse uma tabela.

In [ ]:
con.sql("SELECT * FROM 'landing/pedidos_20240120.csv'").show()

**📝 O que essa célula faz:**

- `con.sql("...")` → manda uma consulta SQL para o DuckDB.
- `.show()` → imprime o resultado formatado.

**🤯 Repare no que NÃO precisamos fazer:**

Não criamos tabela, não declaramos colunas, não fizemos `CREATE TABLE`, não importamos nada. O DuckDB abriu o CSV, **descobriu sozinho** os nomes e os tipos das colunas, e respondeu.

> ⚠️ Note as **aspas simples** em volta do caminho do arquivo, dentro das aspas duplas do Python. O SQL precisa delas.

In [ ]:
# Só as colunas que interessam, com filtro e ordenação
con.sql("""
    SELECT nome, cidade, valor_compra
    FROM 'landing/pedidos_20240120.csv'
    WHERE valor_compra > 500
    ORDER BY valor_compra DESC
""").show()

**📝 O que essa célula faz:**

SQL básico sobre um arquivo CSV solto no disco:

- `SELECT nome, cidade, valor_compra` → escolhe **quais colunas** aparecem.
- `WHERE valor_compra > 500` → escolhe **quais linhas** aparecem (é o `if` do SQL).
- `ORDER BY valor_compra DESC` → ordena do maior para o menor (`DESC` = descendente).

Compare com o módulo de Python: você precisou de um `for`, um `if` e uma lista nova para fazer isso. Aqui são três linhas.

In [ ]:
# Que tipos o DuckDB inferiu?
con.sql("DESCRIBE SELECT * FROM 'landing/pedidos_20240120.csv'").show()

**📝 O que essa célula faz:**

`DESCRIBE` mostra o **schema** — nome e tipo de cada coluna.

Olhe a coluna `data_compra`: veio como `VARCHAR` (texto), não como `DATE`. Por quê? Porque a linha 1008 tem `data_invalida`, que não é uma data. **Um único registro sujo derrubou o tipo da coluna inteira.**

É exatamente o tipo de problema que a camada silver existe para resolver.

## 4. Camada BRONZE

A regra da bronze é uma só: **guardar o dado como chegou**. Sem limpar, sem filtrar, sem corrigir. A única coisa que muda é o formato do arquivo — de CSV para Parquet.

In [ ]:
for camada in ['bronze', 'silver', 'gold']:
    os.makedirs(camada, exist_ok=True)

# Copia o CSV cru para Parquet, sem transformar nada
con.sql("""
    COPY (
        SELECT * FROM 'landing/pedidos_20240120.csv'
    )
    TO 'bronze/pedidos.parquet' (FORMAT PARQUET)
""")

print('✓ Bronze gravado em: bronze/pedidos.parquet')

**📝 O que essa célula faz:**

- O `for` cria as três pastas — é o mesmo `for` sobre lista que você já conhece.
- `COPY ( consulta ) TO 'arquivo' (FORMAT PARQUET)` → o comando do DuckDB para **gravar o resultado de um SELECT em um arquivo**. Guarde esse comando: ele é o que usaremos nas três camadas.

**📦 O que é Parquet?**

Um formato de arquivo **colunar** e **comprimido**, criado para dados analíticos:

| | CSV | Parquet |
|---|---|---|
| Como guarda | linha por linha, como texto | coluna por coluna, comprimido |
| Guarda os tipos? | ❌ tudo vira texto | ✅ o schema vai junto |
| Ler 2 colunas de 50 | lê o arquivo inteiro | lê só as 2 |
| Abre no bloco de notas | sim | não (é binário) |

Parquet é o formato padrão de qualquer Data Lake moderno — AWS, Azure, Databricks, Spark. É o que está **por baixo** do Delta Lake e do Iceberg.

### Vale a pena mesmo? Vamos medir.

Com 8 linhas você não vê diferença. Vamos gerar **500 mil linhas** e comparar.

In [ ]:
# Gera um CSV grande artificialmente
con.sql("""
    COPY (
        SELECT i AS id,
               'cliente_' || i AS nome,
               'SP' AS uf,
               random() * 1000 AS valor
        FROM range(500000) t(i)
    )
    TO 'teste_grande.csv' (FORMAT CSV, HEADER)
""")

# Mesmo conteúdo, em Parquet
con.sql("COPY (SELECT * FROM 'teste_grande.csv') TO 'teste_grande.parquet' (FORMAT PARQUET)")

mb_csv = os.path.getsize('teste_grande.csv') / 1_048_576
mb_pq  = os.path.getsize('teste_grande.parquet') / 1_048_576

print(f'CSV:     {mb_csv:.2f} MB')
print(f'Parquet: {mb_pq:.2f} MB')
print(f'Redução: {(1 - mb_pq / mb_csv) * 100:.0f}%')

**📝 O que essa célula faz:**

- `range(500000) t(i)` → função do DuckDB que gera 500 mil linhas numeradas de 0 a 499.999.
- `'cliente_' || i` → em SQL, `||` junta textos (é o `+` entre strings do Python).
- `os.path.getsize(...) / 1_048_576` → tamanho em bytes convertido para MB. **A mesma conta da primeira célula do curso.**
- `{mb_csv:.2f}` na f-string → formata com 2 casas decimais.

O Parquet costuma ficar cerca de **60% menor**. Em um Data Lake com terabytes, isso é dinheiro de armazenamento e de tráfego de rede todo mês.

In [ ]:
import time

inicio = time.time()
con.sql("SELECT uf, count(*), avg(valor) FROM 'teste_grande.csv' GROUP BY uf").fetchall()
tempo_csv = time.time() - inicio

inicio = time.time()
con.sql("SELECT uf, count(*), avg(valor) FROM 'teste_grande.parquet' GROUP BY uf").fetchall()
tempo_pq = time.time() - inicio

print(f'CSV:     {tempo_csv:.3f}s')
print(f'Parquet: {tempo_pq:.3f}s')
print(f'Parquet foi {tempo_csv / tempo_pq:.0f}x mais rápido')

**📝 O que essa célula faz:**

Mede o tempo das duas consultas.

- `time.time()` → devolve o relógio do computador em segundos. Guardando antes e depois e subtraindo, você tem quanto tempo passou.
- `.fetchall()` → executa e traz o resultado (usamos no lugar de `.show()` porque aqui só queremos medir).

**Por que o Parquet ganha tanto?** No CSV o DuckDB precisa ler e interpretar como texto as 500 mil linhas e **todas** as colunas. No Parquet ele lê apenas as duas colunas de que precisa (`uf` e `valor`), já com o tipo certo.

## 5. Particionamento — organizando por data de ingestão

Até aqui gravamos um arquivo só. Mas em produção chega um arquivo **por dia**. Se você sempre escreve em `bronze/pedidos.parquet`, o de ontem é destruído.

A solução do Lakehouse é o **particionamento**: uma subpasta por data.

In [ ]:
import shutil
shutil.rmtree('bronze', ignore_errors=True)   # limpa o teste anterior

con.sql("""
    COPY (
        SELECT *, '2024-01-20'::DATE AS data_ingestao
        FROM read_csv('landing/pedidos_20240120.csv',
                      types={'data_compra': 'VARCHAR'})
    )
    TO 'bronze' (FORMAT PARQUET,
                 PARTITION_BY (data_ingestao),
                 OVERWRITE_OR_IGNORE)
""")

# Mostra a estrutura de pastas que foi criada
for pasta, _, arquivos in os.walk('bronze'):
    for arq in arquivos:
        print(os.path.join(pasta, arq))

**📝 O que essa célula faz:**

- `shutil.rmtree('bronze', ignore_errors=True)` → apaga a pasta `bronze` e tudo dentro dela. `ignore_errors=True` evita erro caso ela não exista.
- `SELECT *, '2024-01-20'::DATE AS data_ingestao` → pega todas as colunas do arquivo **e acrescenta uma nova**, com a data em que o dado entrou. `::DATE` converte o texto para o tipo data.
- `read_csv(..., types={'data_compra': 'VARCHAR'})` → em vez de deixar o DuckDB adivinhar, **mandamos** ler `data_compra` como texto. Na bronze isso é a atitude certa: o dado bruto não é confiável, e forçar um tipo aqui faria a leitura quebrar quando chegasse lixo.
- `PARTITION_BY (data_ingestao)` → em vez de um arquivo só, o DuckDB cria **uma subpasta por valor** dessa coluna.
- `OVERWRITE_OR_IGNORE` → permite sobrescrever uma partição que já exista, sem reclamar.

**🗂️ Repare no nome da pasta criada: `bronze/data_ingestao=2024-01-20/`**

Esse padrão `coluna=valor` no nome da pasta se chama **Hive partitioning**. É um padrão de mercado que Spark, Athena, BigQuery e Trino entendem nativamente. Quando você consulta `WHERE data_ingestao = '2024-01-20'`, a engine lê **só aquela pasta** e ignora todo o resto — é a otimização mais importante de um Data Lake.

> No lab `34.Pipeline-Integrado` do repositório, o caminho é exatamente esse: `s3a://bronze/randomuser/ingestion_date=.../`

### Chegou o arquivo do dia seguinte

Agora o cenário realista: no dia 21 chega um novo arquivo. Ele traz dois clientes novos **e uma correção** do cliente 1003, cujo valor mudou de 4200 para 5000.

In [ ]:
dados_dia_21 = """id,nome,email,cidade,valor_compra,data_compra
1003,Carla Dias,carla@email.com,Belo Horizonte,5000.00,21/01/2024
1009,Heitor Paz,heitor@email.com,Curitiba,2300.00,21/01/2024
1010,Ivone Sá,ivone@email.com,São Paulo,150.00,21/01/2024
"""

with open('landing/pedidos_20240121.csv', 'w', encoding='utf-8') as f:
    f.write(dados_dia_21)

# Mesma ingestão, outra data de partição
con.sql("""
    COPY (
        SELECT *, '2024-01-21'::DATE AS data_ingestao
        FROM read_csv('landing/pedidos_20240121.csv',
                      types={'data_compra': 'VARCHAR'})
    )
    TO 'bronze' (FORMAT PARQUET,
                 PARTITION_BY (data_ingestao),
                 OVERWRITE_OR_IGNORE)
""")

for pasta, _, arquivos in os.walk('bronze'):
    for arq in arquivos:
        print(os.path.join(pasta, arq))

**📝 O que essa célula faz:**

Grava o segundo arquivo na bronze, exatamente com o mesmo comando — só muda a data.

Resultado: **duas pastas**, uma por dia. Nada foi sobrescrito. O histórico está preservado, que é a razão de existir da bronze.

In [ ]:
# Lendo TODAS as partições de uma vez
con.sql("""
    SELECT data_ingestao, count(*) AS registros
    FROM read_parquet('bronze/**/*.parquet', hive_partitioning=true)
    GROUP BY data_ingestao
    ORDER BY data_ingestao
""").show()

**📝 O que essa célula faz:**

- `read_parquet('bronze/**/*.parquet')` → lê **todos** os arquivos Parquet dentro de `bronze/`, em qualquer subpasta. O `**` significa "qualquer nível de pasta" e o `*` significa "qualquer nome de arquivo". Isso se chama *glob*.
- `hive_partitioning=true` → diz ao DuckDB para ler o nome das pastas (`data_ingestao=2024-01-20`) e transformar em **coluna**. Sem isso, essa coluna simplesmente não existiria.

**Essa é a ideia central de um Data Lake:** você não trabalha com "um arquivo", você trabalha com **uma pasta que se comporta como uma tabela**. Chegou arquivo novo? Ele entra na consulta automaticamente, sem ninguém mexer no código.

In [ ]:
# Existe algum id que aparece em mais de uma carga?
con.sql("""
    SELECT id, count(*) AS vezes
    FROM read_parquet('bronze/**/*.parquet', hive_partitioning=true)
    GROUP BY id
    HAVING count(*) > 1
""").show()

**📝 O que essa célula faz:**

Procura ids duplicados.

- `GROUP BY id` → agrupa por id.
- `HAVING count(*) > 1` → filtra **depois** do agrupamento, mantendo só os grupos com mais de um registro. (`WHERE` filtra linhas antes de agrupar; `HAVING` filtra grupos depois.)

**Resultado: o id 1003 aparece 2 vezes** — uma na carga do dia 20 (valor 4200) e outra na do dia 21 (valor 5000).

Isso **não é um bug da bronze** — é o comportamento correto dela: a bronze guarda tudo o que chegou, inclusive as versões antigas. Quem resolve a duplicata é a próxima camada.

## 6. Camada SILVER — limpar, deduplicar, validar

A silver tem três tarefas:

1. **Deduplicar** — uma linha por cliente, ficando com a versão mais recente
2. **Limpar e tipar** — `trim`, `lower`, texto → data
3. **Validar** — descartar o que não atende às regras de negócio

Vamos por partes. Primeiro a deduplicação.

In [ ]:
con.sql("""
    SELECT id, nome, valor_compra, data_ingestao, rn
    FROM (
        SELECT *,
               row_number() OVER (PARTITION BY id ORDER BY data_ingestao DESC) AS rn
        FROM read_parquet('bronze/**/*.parquet', hive_partitioning=true)
    )
    WHERE id = 1003
""").show()

**📝 O que essa célula faz:**

Mostra as duas versões do cliente 1003, numeradas.

**🔢 `row_number() OVER (PARTITION BY id ORDER BY data_ingestao DESC)`**

Traduzindo para português: *"para cada `id`, ordene os registros da data de ingestão mais nova para a mais velha e numere-os 1, 2, 3..."*

- `PARTITION BY id` → a numeração **reinicia** a cada id diferente
- `ORDER BY data_ingestao DESC` → mais recente primeiro
- `row_number()` → dá o número

Resultado: a versão mais nova sempre recebe `rn = 1`. Então **filtrar `WHERE rn = 1` fica com a versão mais recente de cada cliente** e descarta as antigas.

Esse padrão tem nome — *deduplicação por janela* — e é uma das coisas que você mais vai escrever na vida como engenheiro de dados.

In [ ]:
con.sql("""
    COPY (
        SELECT
            id,
            trim(nome)                                   AS nome,
            lower(trim(email))                           AS email,
            cidade,
            valor_compra,
            try_strptime(data_compra, '%d/%m/%Y')::DATE  AS data_compra,
            data_ingestao,
            CASE
                WHEN valor_compra >= 2000 THEN 'vip'
                WHEN valor_compra >= 500  THEN 'regular'
                ELSE 'ocasional'
            END                                          AS classificacao
        FROM (
            SELECT * FROM (
                SELECT *,
                       row_number() OVER (PARTITION BY id ORDER BY data_ingestao DESC) AS rn
                FROM read_parquet('bronze/**/*.parquet', hive_partitioning=true)
            )
            WHERE rn = 1
        )
        WHERE nome         IS NOT NULL
          AND email        IS NOT NULL
          AND valor_compra IS NOT NULL
          AND valor_compra >= 0
          AND try_strptime(data_compra, '%d/%m/%Y') IS NOT NULL
    )
    TO 'silver/pedidos.parquet' (FORMAT PARQUET)
""")

con.sql("SELECT * FROM 'silver/pedidos.parquet' ORDER BY id").show()

**📝 O que essa célula faz:**

Junta as três tarefas da silver em uma consulta. Compare linha a linha com o que você escreveu em Python no módulo anterior:

| No pipeline Python | Aqui em SQL |
|---|---|
| `def limpar_texto(t): return t.strip()` | `trim(nome)` |
| `texto.lower()` | `lower(email)` |
| `def converter_data(t): ...` | `try_strptime(data_compra, '%d/%m/%Y')` |
| `def classificar_cliente(v): if/elif/else` | `CASE WHEN ... THEN ... ELSE ... END` |
| `def validar_registro(rec): if not rec[...]` | `WHERE ... IS NOT NULL AND ...` |
| (não existia) | `row_number()` para deduplicar |

**🛡️ `try_strptime` — o `try/except` do SQL**

`strptime` converte texto em data seguindo um formato (`%d/%m/%Y` = dia/mês/ano). A versão com `try_` **não quebra** quando o texto não é uma data: devolve `NULL`. É a mesma ideia do `try/except` que você viu em Python.

**`::DATE`** é o jeito curto de dizer "converta para data" — equivalente a `CAST(... AS DATE)`.

**Olhe o resultado com atenção:**

- **11 registros brutos → 5 válidos**
- O cliente **1003 aparece uma vez só, com valor 5000** — a versão corrigida do dia 21 venceu
- Os 5 registros problemáticos do dia 20 sumiram

### Quarentena — nunca jogue o dado ruim fora

O dado rejeitado é a sua evidência de que existe problema na origem. Ele vai para um arquivo separado, com o motivo.

In [ ]:
con.sql("""
    COPY (
        SELECT
            id, nome, email, valor_compra, data_compra, data_ingestao,
            CASE
                WHEN nome         IS NULL THEN 'nome ausente'
                WHEN email        IS NULL THEN 'email ausente'
                WHEN valor_compra IS NULL THEN 'valor ausente'
                WHEN valor_compra < 0     THEN 'valor negativo'
                ELSE 'data inválida'
            END AS motivo_rejeicao
        FROM read_parquet('bronze/**/*.parquet', hive_partitioning=true)
        WHERE nome         IS NULL
           OR email        IS NULL
           OR valor_compra IS NULL
           OR valor_compra < 0
           OR try_strptime(data_compra, '%d/%m/%Y') IS NULL
    )
    TO 'silver/pedidos_rejeitados.parquet' (FORMAT PARQUET)
""")

con.sql("SELECT id, nome, motivo_rejeicao FROM 'silver/pedidos_rejeitados.parquet'").show()

**📝 O que essa célula faz:**

Pega o **oposto** do filtro anterior — note que os `AND` viraram `OR` — e grava em arquivo separado com o motivo da rejeição.

**Por que `OR` e não `AND`?** Para ser aprovado, o registro precisa passar em **todas** as regras (`AND`). Para ser rejeitado, basta falhar em **uma** (`OR`).

Esse arquivo é a sua **quarentena**. É o que você manda para a área de negócio quando perguntam por que o relatório tem menos pedidos do que o sistema de origem.

## 7. Camada GOLD — a pergunta do negócio respondida

A gold não tem dado técnico: tem **resposta**. Uma tabela por pergunta.

In [ ]:
con.sql("""
    COPY (
        SELECT
            classificacao,
            count(*)                     AS qtd_pedidos,
            round(sum(valor_compra), 2)  AS receita_total,
            round(avg(valor_compra), 2)  AS ticket_medio
        FROM 'silver/pedidos.parquet'
        GROUP BY classificacao
        ORDER BY receita_total DESC
    )
    TO 'gold/receita_por_classificacao.parquet' (FORMAT PARQUET)
""")

con.sql("SELECT * FROM 'gold/receita_por_classificacao.parquet'").show()

**📝 O que essa célula faz:**

- `GROUP BY classificacao` → agrupa por categoria de cliente.
- `count(*)`, `sum(...)`, `avg(...)` → contam, somam e tiram média **dentro de cada grupo**.
- `round(..., 2)` → arredonda para 2 casas.
- `ORDER BY receita_total DESC` → da maior receita para a menor.

**Por que gravar em arquivo, se é só uma consulta?**

Porque em produção essa tabela é lida por dezenas de dashboards, várias vezes ao dia. Calcular uma vez e salvar é mais barato e mais rápido do que reprocessar a silver a cada abertura de relatório. Isso se chama **materializar** a camada gold.

In [ ]:
# Alternativa: VIEW — sempre atualizada, calculada na hora
con.sql("""
    CREATE OR REPLACE VIEW vw_receita_cidade AS
    SELECT cidade,
           count(*)                    AS qtd_pedidos,
           round(sum(valor_compra), 2) AS receita
    FROM 'silver/pedidos.parquet'
    GROUP BY cidade
    ORDER BY receita DESC
""")

con.sql("SELECT * FROM vw_receita_cidade").show()

**📝 O que essa célula faz:**

Cria uma **VIEW**: uma consulta salva com nome. Ela não guarda dado nenhum — toda vez que você faz `SELECT * FROM vw_receita_cidade`, o DuckDB roda a consulta original de novo.

| | Tabela materializada (arquivo) | VIEW |
|---|---|---|
| Guarda dados | ✅ sim | ❌ não, é só a consulta |
| Sempre atualizada | ❌ até o próximo processamento | ✅ sempre |
| Custo de leitura | baixo | recalcula toda vez |
| Quando usar | dashboard com muito acesso | consulta pontual, dado que muda toda hora |

`CREATE OR REPLACE` cria a view ou substitui, se já existir — evita erro ao rodar a célula duas vezes.

## 8. Python e SQL conversando

O DuckDB não substitui o Python — **complementa**. SQL faz o que SQL faz bem (filtrar, juntar, agregar). Python faz o resto (orquestrar, chamar API, tratar erro, gravar log, mandar alerta).

In [ ]:
resultado = con.sql("""
    SELECT classificacao, receita_total
    FROM 'gold/receita_por_classificacao.parquet'
""").fetchall()

print('Tipo do retorno:', type(resultado))
print()

# Daqui pra frente é Python puro — o mesmo for que você já conhece
for classificacao, receita in resultado:
    print(f'{classificacao:>10}: R$ {receita:>10,.2f}')

**📝 O que essa célula faz:**

- `.fetchall()` → traz o resultado para dentro do Python como **lista de tuplas**. Uma tupla é como uma lista, só que não pode ser modificada.
- `for classificacao, receita in resultado:` → percorre a lista **já separando** os dois valores de cada linha em duas variáveis. Isso se chama *desempacotamento*.
- `{classificacao:>10}` alinha o texto à direita em 10 caracteres. `{receita:>10,.2f}` faz o mesmo com o número, com separador de milhar e 2 casas.

**É esse o padrão do dia a dia:** o SQL faz o trabalho pesado sobre milhões de linhas em arquivo, e o Python recebe o resultado já reduzido a poucas linhas e decide o que fazer com ele.

## 9. O pipeline inteiro em uma função

Juntando o que você aprendeu sobre funções com tudo o que vimos aqui:

In [ ]:
def ingerir_bronze(arquivo_landing, data_ingestao):
    """Grava um arquivo da landing na bronze, particionado por data."""
    con.sql(f"""
        COPY (
            SELECT *, '{data_ingestao}'::DATE AS data_ingestao
            FROM read_csv('{arquivo_landing}', types={{'data_compra': 'VARCHAR'}})
        )
        TO 'bronze' (FORMAT PARQUET, PARTITION_BY (data_ingestao), OVERWRITE_OR_IGNORE)
    """)
    total = con.sql("SELECT count(*) FROM read_parquet('bronze/**/*.parquet', hive_partitioning=true)").fetchone()[0]
    print(f'  🥉 Bronze: {total} registros acumulados')
    return total


def processar_silver():
    """Deduplica, limpa e valida a bronze inteira, gravando a silver."""
    con.sql("""
        COPY (
            SELECT id, trim(nome) AS nome, lower(trim(email)) AS email, cidade,
                   valor_compra,
                   try_strptime(data_compra, '%d/%m/%Y')::DATE AS data_compra,
                   data_ingestao,
                   CASE WHEN valor_compra >= 2000 THEN 'vip'
                        WHEN valor_compra >= 500  THEN 'regular'
                        ELSE 'ocasional' END AS classificacao
            FROM (
                SELECT * FROM (
                    SELECT *, row_number() OVER (PARTITION BY id ORDER BY data_ingestao DESC) AS rn
                    FROM read_parquet('bronze/**/*.parquet', hive_partitioning=true)
                ) WHERE rn = 1
            )
            WHERE nome IS NOT NULL AND email IS NOT NULL
              AND valor_compra IS NOT NULL AND valor_compra >= 0
              AND try_strptime(data_compra, '%d/%m/%Y') IS NOT NULL
        ) TO 'silver/pedidos.parquet' (FORMAT PARQUET)
    """)
    total = con.sql("SELECT count(*) FROM 'silver/pedidos.parquet'").fetchone()[0]
    print(f'  🥈 Silver: {total} registros válidos e únicos')
    return total


def processar_gold():
    """Gera a tabela agregada para o negócio."""
    con.sql("""
        COPY (
            SELECT classificacao, count(*) AS qtd_pedidos,
                   round(sum(valor_compra), 2) AS receita_total
            FROM 'silver/pedidos.parquet'
            GROUP BY classificacao ORDER BY receita_total DESC
        ) TO 'gold/receita_por_classificacao.parquet' (FORMAT PARQUET)
    """)
    print('  🥇 Gold: tabela de receita gerada')


def executar_lakehouse(arquivo_landing, data_ingestao):
    """Executa bronze → silver → gold e devolve um resumo da execução."""
    print(f'🚀 Processando {arquivo_landing}')
    bruto   = ingerir_bronze(arquivo_landing, data_ingestao)
    validos = processar_silver()
    processar_gold()

    descartados = bruto - validos
    print(f'\n✅ Concluído — {descartados} registros descartados (rejeitados + duplicados)')
    return {'bruto': bruto, 'validos': validos, 'descartados': descartados}


resumo = executar_lakehouse('landing/pedidos_20240121.csv', '2024-01-21')
print()
print('Resumo:', resumo)

**📝 O que essa célula faz:**

Empacota o pipeline em **quatro funções pequenas** em vez de uma gigante — uma por camada, mais uma que orquestra as três. É o mesmo princípio de funções que você viu no módulo de Python: cada função faz **uma coisa**, tem nome que descreve o que faz, e pode ser testada sozinha.

- `f"""...{arquivo_landing}..."""` → **f-string de três aspas**: permite escrever SQL em várias linhas e ainda inserir variáveis Python com `{}`. É assim que a função serve para qualquer arquivo.
- `{{'data_compra': 'VARCHAR'}}` → dentro de uma f-string, `{{` e `}}` são a forma de escrever uma chave literal `{` e `}`. Sem duplicar, o Python acharia que é uma variável.
- `.fetchone()[0]` → quando a consulta devolve um único número (um `count`), `.fetchone()` traz a primeira linha e `[0]` pega o primeiro valor dela.
- `return {...}` → devolve um dicionário com os números da execução, para quem chamou poder gravar em log ou disparar alerta.

**Repare:** essa é a estrutura de um job de produção de verdade. Trocando `con.sql` por `spark.sql`, o desenho é idêntico. É por isso que aprender aqui vale para lá.

## 🏋️ Exercício — Sua própria camada gold

A área de negócio pediu três coisas. Complete o código.

**1.** Receita **por cidade**, da maior para a menor.
**2.** Um **resumo geral**: total de pedidos válidos, receita total e o maior pedido (dica: `max()`).
**3.** Quantos registros foram **rejeitados por cada motivo** (use o arquivo de quarentena).

> 💡 Copie a estrutura do `COPY (...) TO ... (FORMAT PARQUET)` da seção 7 e troque só o `SELECT`.

In [ ]:
# 1. Receita por cidade
con.sql("""
    COPY (
        SELECT
            -- seu código aqui
        FROM 'silver/pedidos.parquet'
        -- GROUP BY ...
        -- ORDER BY ...
    )
    TO 'gold/receita_por_cidade.parquet' (FORMAT PARQUET)
""")

con.sql("SELECT * FROM 'gold/receita_por_cidade.parquet'").show()

In [ ]:
# 2. Resumo geral — uma única linha de resultado, sem GROUP BY
con.sql("""
    COPY (
        SELECT
            -- seu código aqui
        FROM 'silver/pedidos.parquet'
    )
    TO 'gold/resumo_geral.parquet' (FORMAT PARQUET)
""")

con.sql("SELECT * FROM 'gold/resumo_geral.parquet'").show()

In [ ]:
# 3. Rejeições por motivo
con.sql("""
    SELECT
        -- seu código aqui
    FROM 'silver/pedidos_rejeitados.parquet'
    -- GROUP BY ...
""").show()

## 10. Onde isso vira Spark, Delta e MinIO

O que você construiu hoje é a **versão de bancada** do laboratório `34.Pipeline-Integrado` do repositório. As ideias são idênticas; muda a escala e a infraestrutura:

| Conceito | Nesta aula | No lab Pipeline-Integrado (produção) |
|---|---|---|
| Onde ficam os arquivos | pastas `bronze/`, `silver/`, `gold/` | buckets no **MinIO** (`s3a://bronze/...`) — em produção, S3 |
| Formato dos arquivos | **Parquet** | **Delta Lake** = Parquet + um log de transações |
| Engine de processamento | **DuckDB** (uma máquina) | **Spark** (cluster de várias máquinas) |
| Linguagem | SQL + Python | PySpark + SQL |
| Entrada dos dados | arquivo na `landing/` | API + **Kafka** (streaming) |
| Particionamento | `PARTITION_BY (data_ingestao)` | `partitionBy("ingestion_date")` — mesma pasta `coluna=valor` |
| Atualizar sem duplicar | `row_number()` + reescrita | `MERGE` (upsert) nativo da tabela Delta |
| Ver versões antigas | ❌ não tem | ✅ **Time Travel** (`versionAsOf`) |
| Evoluir o schema | reescrever o arquivo | `mergeSchema = true` |

**A deduplicação que você fez na seção 6 é exatamente o problema que o `MERGE` do Delta Lake resolve.** Você precisou de uma janela `row_number()` e de reescrever a silver inteira. No Delta, é uma instrução:

```python
delta_table.alias("silver").merge(
    novos_dados.alias("bronze"), "silver.id = bronze.id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
```

Entender **por que** esse comando existe é mais valioso do que decorá-lo — e agora você entende, porque sentiu o problema na mão.

### Quando usar cada um?

- **DuckDB**: dados que cabem em uma máquina — o que hoje vai tranquilamente até dezenas de GB. Protótipos, análises, validação de lógica de transformação, jobs pequenos e médios. Custo próximo de zero, sobe em 10 segundos.
- **Spark + Delta**: dados que **não** cabem em uma máquina, ou quando você precisa de transações ACID, time travel e streaming de verdade.

> 💡 A recomendação prática: **comece no DuckDB**. A maioria dos pipelines do dia a dia nunca precisa de cluster — e uma lógica validada no DuckDB é fácil de traduzir para Spark depois. O erro caro é o contrário: subir um cluster para processar 200 MB.

## Resumo

| Conceito | O que guardar |
|---|---|
| **Lakehouse** | Data Lake (arquivos baratos) com qualidades de Data Warehouse (schema, SQL, transações) |
| **landing** | onde o arquivo aterrissa, cru, vindo da origem |
| **bronze** | dado como chegou — só muda o formato. É a rede de segurança |
| **silver** | deduplicado, limpo, tipado, validado. Uma linha por entidade |
| **gold** | agregado, pronto para o negócio. Uma tabela por pergunta |
| **quarentena** | dado rejeitado nunca é descartado, é separado com o motivo |
| **Parquet** | colunar e comprimido: ~60% menor e muito mais rápido que CSV |
| **Particionamento** | pasta `coluna=valor`; a engine lê só a pasta que interessa |
| **DuckDB** | engine SQL sem servidor, lê arquivos direto do disco |
| **`COPY (...) TO ...`** | grava o resultado de um SELECT em arquivo |
| **`read_parquet('**/*.parquet')`** | trata uma pasta inteira como se fosse uma tabela |
| **`row_number() OVER (...)`** | deduplicação: fica com a versão mais recente |
| **`try_strptime`** | converte texto em data sem quebrar — o `try/except` do SQL |
| **`.fetchall()`** | traz o resultado do SQL para dentro do Python |

### A ideia que fica

O pipeline da Aula 6 e o desta aula fazem **a mesma coisa**. O que muda é a ferramenta.

Python orquestra. SQL transforma. Camadas organizam. Trocar DuckDB por Spark, ou Parquet por Delta, muda a **escala** — não muda o **raciocínio**.

**Próximo passo:** o laboratório `34.Pipeline-Integrado`, que resolve esses mesmos problemas com Spark, Delta Lake, Kafka e MinIO.